# 05 · 数据层：Phenopacket Store 的下载、校验、解析（路线第 2 步第 1 块）

**JD 对应**：「数据管线与数据质量」「Python 工程化：数据模型（pydantic）、命令行工具、单元测试」。

**本章新东西的分类**（A 通用标准 / B 领域内多选一 / C 只为你学）：

| 东西 | 类 | 说明 |
|---|---|---|
| 下载 + sha256 校验 + 幂等 | A | 所有正经数据管线都这么做；你的 qafi-data-platform 里 `_manifest.json` 是同一件事 |
| pydantic 数据模型 | A | Python 里定义"一条记录长什么样"的事实标准 |
| typer 命令行 | B | 同类还有 click、argparse；typer 最省事，FastAPI 同一作者 |
| 开发集 / 留出集按论文切分 | A（评测方法学） | 机器学习和临床研究都用；按什么单位切是关键 |
| 用合成的小数据做单元测试 | A | 测试不碰 19 MB 真数据、不联网，CI 才跑得动 |

## 做了什么

```bash
uv add pydantic pydantic-settings typer httpx       # 数据模型、配置、命令行、下载
uv run raredx data fetch                             # 下载 0.1.27 发布包到 data/，校验 sha256
uv run raredx data stats                             # 解析 10,377 个 JSON，打印统计
```

| 文件 | 作用 |
|---|---|
| `src/raredx/config.py` | 配置入口：环境变量 `RAREDX_*` 或 `.env`。目前只有一项 `data_dir` |
| `src/raredx/data/phenopackets.py` | 下载 + 校验、解析成 `Case` 对象、按 PMID 切分开发集/留出集 |
| `src/raredx/cli.py` | `raredx data fetch` / `raredx data stats`。薄：只调包里的函数然后打印 |
| `tests/data/test_phenopackets.py` | 在内存里造一个 3 例的小数据包，测解析、过滤、切分、校验 |
| `.env.example` | 配置模板；真正的 `.env` 不进 git |

一条病例（`Case`）长这样：`id`、`cohort`（数据包里的文件夹，通常是基因名）、`hpo_ids` / `hpo_labels`
（作者观察到的症状）、`disease_id`（OMIM 编号）、`disease_label`、`gene`、`variants`、`pmid`、`sex`、`split`。

## 工业界叫什么

| 术语 | 大白话 |
|---|---|
| checksum / manifest（校验和 / 清单） | 文件的指纹。下载下来算一遍，和记录的对不上就删掉，不用 |
| idempotent（幂等） | 同一条命令跑一次和跑十次结果一样：文件已存在且校验通过就不重新下载 |
| schema / data model（数据模型） | 一条记录有哪些字段、各是什么类型。pydantic 在读入时就检查，错的进不来 |
| fixture（测试夹具） | 测试前准备的小环境。这里是 `tmp_path` 里现造的一个 zip |
| design / held-out split（开发集 / 留出集） | 开发时只看开发集；留出集只在设计定稿后跑一次，看过就"用掉"了 |
| raw data layer（原始数据层） | `data/raw/<来源>/<版本>/` 原样存放，不改一个字节；派生数据另放 |

## 为什么

- **数据不进 git，但必须可复现。** 所以是"下载脚本 + 校验和 + 版本号"，而不是把 19 MB 塞进仓库。
  任何人 clone 后 `raredx data fetch` 得到的是字节级相同的文件（旧仓库那份的 sha256 和官方的一致，我比过）。
- **`excluded` 这个标记不过滤就全错。** 数据里第一个样例 37 个症状，33 个是"查过但没有"。
  旧仓库的加载器处理对了，这里照搬。
- **按论文切分，不按病例。** 同一篇论文里的几个病人常是一家人，措辞也一样；一个放开发集一个放留出集，
  等于提前看过答案。用 PMID 的哈希决定去向，固定且没人能挑。代价是两边不均（6,014 / 4,360），
  因为大队列论文整篇落在一边——可以接受，比均匀但泄漏好。
- **测试不碰真数据。** 真数据 19 MB、要联网；CI 上没有。合成 3 例覆盖了正常、全部 excluded、没基因三种情况。

## ✏️ 你来验证

In [1]:
import os, subprocess
ROOT = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(ROOT)
os.environ.update(TERM="dumb", NO_COLOR="1", GH_SPINNER_DISABLED="1", GH_PAGER="cat")
print("当前目录:", ROOT)

当前目录: /Users/pipi/Projects/QAFI_Paper/hearing_agent/rare-disease-agent


下载（已存在就只校验，秒回）：

In [2]:
!uv run raredx data fetch

ok  data/raw/phenopacket-store/0.1.27/all_phenopackets.zip  (release 0.1.27, sha256 verified)


统计：

In [3]:
!uv run raredx data stats

cases                                    10374
skipped                                  {'no_phenotype': 3}
with a causal gene                       10305
distinct diseases (OMIM)                 780
distinct genes                           699
distinct source papers (PMID)            1733
HPO terms per case, median (min–max)     7 (1–66)
split design / heldout                   6014 / 4360


看一条真实病例长什么样（第一条）：

In [4]:
from pathlib import Path
from raredx.data import phenopackets as pp

cases, skipped = pp.load(pp.archive_path(Path("data")))
print(cases[0].model_dump_json(indent=2))

{
  "id": "PMID_15266616_100",
  "cohort": "11q_terminal_deletion",
  "hpo_ids": [
    "HP:0000348",
    "HP:0011220",
    "HP:0000337",
    "HP:0002553",
    "HP:0010487",
    "HP:0001763",
    "HP:0000078"
  ],
  "hpo_labels": [
    "High forehead",
    "Prominent forehead",
    "Broad forehead",
    "Highly arched eyebrow",
    "Small hypothenar eminence",
    "Pes planus",
    "Abnormality of the genital system"
  ],
  "disease_id": "OMIM:147791",
  "disease_label": "Jacobsen syndrome",
  "gene": null,
  "variants": [],
  "pmid": "PMID:15266616",
  "sex": "FEMALE",
  "split": "heldout"
}


为什么开发集/留出集不均：几篇大队列论文各贡献上百例，整篇落在一边。看贡献最多的 8 篇：

In [5]:
from collections import Counter

per_paper = Counter(c.pmid for c in cases)
for pmid, n in per_paper.most_common(8):
    print(f"{pmid:<16} {n:>4} 例   → {pp.split_of(pmid)}")

PMID:35190816     462 例   → design
PMID:36446582     333 例   → design
PMID:29290338     157 例   → design
PMID:31021519     157 例   → design
PMID:26178382     136 例   → design
PMID:39013458     125 例   → design
PMID:37843397     115 例   → design
PMID:25802881     103 例   → design


数据层自己的测试：

In [6]:
!uv run pytest tests/data -q

......                                                                   [100%]


## 思考

为什么开发集 / 留出集要按论文（PMID）切，而不是简单地随机按病例切？先想，再看下一格。

**答案**：同一篇论文里的病人往往是同一个家系、同一位作者的措辞、同一种检查套路。随机按病例切，
留出集里的病例在开发集里有"兄弟姐妹"，模型或规则看过兄弟就能答对妹妹——留出集的分数会虚高，
而且你发现不了。按论文切，整篇论文只能在一边。这和临床研究里"按患者而不是按样本切分"是同一个道理。